# Fine-Tuning with LoRA & DPO — Full Pipeline

This notebook runs the entire fine-tuning pipeline on Google Colab:

1. **Setup** — Mount Google Drive, install dependencies
2. **Data Preparation** — Generate SFT and DPO training data
3. **SFT Training** — Supervised fine-tuning with QLoRA on Qwen 3 8B
4. **DPO Training** — Preference optimization on top of SFT checkpoint
5. **Evaluation** — Compare base model vs fine-tuned model
6. **Manual Testing** — Interactive inference to test the model

**Requirements:** Select a **T4 GPU** runtime (Runtime > Change runtime type > T4 GPU)

---
## 0. Verify GPU

In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


---
## 1. Mount Google Drive & Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# UPDATE THIS PATH if your folder is in a different location on Google Drive
PROJECT_DIR = "/content/drive/MyDrive/fine-tuning-lora-dpo"

import os
assert os.path.exists(PROJECT_DIR), f"Project not found at {PROJECT_DIR}. Update PROJECT_DIR above."
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

# Make PROJECT_DIR available to all shell (!) commands
%env PROJECT_DIR={PROJECT_DIR}

Working directory: /content/drive/MyDrive/fine-tuning-lora-dpo
total 462
-rw------- 1 root root  11197 May 13 18:52 AI-PORTFOLIO-GUIDE.md
drwx------ 2 root root   4096 Jul 20 21:23 app.egg-info
-rw------- 1 root root 413758 Jul 20 21:37 colab_notebook.ipynb
drwx------ 2 root root   4096 Jul 15 06:08 configs
drwx------ 2 root root   4096 Jul 20 21:23 data
-rw------- 1 root root    332 May 13 13:48 Dockerfile
-rw------- 1 root root    112 May 13 13:57 .dockerignore
drwx------ 2 root root   4096 Jul 15 06:08 .git
drwx------ 2 root root   4096 Jul 15 06:08 .github
-rw------- 1 root root     80 May 13 12:34 .gitignore
drwx------ 4 root root   4096 Jul 15 07:01 models
-rw------- 1 root root    202 May 13 18:56 pyproject.toml
drwx------ 3 root root   4096 Jul 15 06:08 .pytest_cache
-rw------- 1 root root   3349 May 13 18:52 README.md
-rw------- 1 root root    209 May 13 12:33 requirements.txt
drwx------ 2 root root   4096 Jul 15 06:08 .ruff_cache
drwx------ 3 root root   4096 Jul 15 06:08 src

In [4]:
# Install dependencies
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
!pip install -q -e .

/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for app (pyproject.toml) ... done


---
## 2. Prepare Training Data

In [5]:
%cd {PROJECT_DIR}
!python src/prepare_data.py

/content
Saved function definitions to data/function_definitions.json
Saved 16 examples to data/sft_train.jsonl
Saved 4 examples to data/sft_eval.jsonl
Saved 6 examples to data/dpo_train.jsonl
Saved 2 examples to data/dpo_eval.jsonl

Total SFT examples: 20 (train: 16, eval: 4)
Total DPO examples: 8 (train: 6, eval: 2)

Note: For production, augment with the full Glaive dataset:
  huggingface-cli download glaiveai/glaive-function-calling-v2


In [6]:
# Verify the generated data
import json
os.chdir(PROJECT_DIR)

for path in ["data/sft_train.jsonl", "data/sft_eval.jsonl", "data/dpo_train.jsonl", "data/dpo_eval.jsonl"]:
    with open(path) as f:
        lines = f.readlines()
    # Validate each line is valid JSON
    for i, line in enumerate(lines):
        json.loads(line)
    print(f"{path}: {len(lines)} examples (all valid JSON)")

# Show a sample SFT example
print("\n--- Sample SFT example ---")
with open("data/sft_train.jsonl") as f:
    sample = json.loads(f.readline())
print(json.dumps(sample, indent=2))

# Show a sample DPO example
print("\n--- Sample DPO example ---")
with open("data/dpo_train.jsonl") as f:
    sample = json.loads(f.readline())
print(json.dumps(sample, indent=2))

data/sft_train.jsonl: 16 examples (all valid JSON)
data/sft_eval.jsonl: 4 examples (all valid JSON)
data/dpo_train.jsonl: 6 examples (all valid JSON)
data/dpo_eval.jsonl: 2 examples (all valid JSON)

--- Sample SFT example ---
{
  "instruction": "You have access to the following functions. Call the appropriate function based on the user's request.",
  "input": "What's the current price of Apple stock?",
  "output": "{\"function\": \"get_stock_price\", \"arguments\": {\"ticker\": \"AAPL\", \"exchange\": \"NASDAQ\"}}"
}

--- Sample DPO example ---
{
  "prompt": "You have access to financial tools. User says: 'What is Apple's stock price?'",
  "chosen": "{\"function\": \"get_stock_price\", \"arguments\": {\"ticker\": \"AAPL\", \"exchange\": \"NASDAQ\"}}",
  "rejected": "{\"function\": \"get_stock_price\", \"arguments\": {\"ticker\": \"Apple\"}}"
}


---
## 3. Generate Training Data

Two data sources combined:
1. **Augmented custom data** — 320 examples generated from financial function templates
2. **Glaive function-calling dataset** — filtered to only keep valid JSON function calls

In [7]:
# Generate augmented training data — many diverse queries mapped to correct JSON function calls
import json, random
from pathlib import Path

random.seed(42)

INSTRUCTION = "You have access to the following functions. Call the appropriate function based on the user's request."

# Templates: (query_template, function_call_dict)
# Each template has {var} placeholders that get filled with random values
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "META", "NVDA", "JPM", "GS", "BAC", "WMT", "DIS", "NFLX", "AMD", "INTC", "CRM", "ORCL", "PYPL", "SQ", "COIN"]
COMPANY_NAMES = {"AAPL": "Apple", "MSFT": "Microsoft", "GOOGL": "Google", "AMZN": "Amazon", "TSLA": "Tesla", "META": "Meta", "NVDA": "NVIDIA", "JPM": "JPMorgan", "GS": "Goldman Sachs", "BAC": "Bank of America", "WMT": "Walmart", "DIS": "Disney", "NFLX": "Netflix", "AMD": "AMD", "INTC": "Intel", "CRM": "Salesforce", "ORCL": "Oracle", "PYPL": "PayPal", "SQ": "Block", "COIN": "Coinbase"}
EXCHANGES = ["NYSE", "NASDAQ"]
REPORT_TYPES = ["10-K", "10-Q", "8-K", "earnings"]
YEARS = [2023, 2024, 2025]
QUARTERS = [1, 2, 3, 4]
RATIO_TYPES = ["pe_ratio", "debt_to_equity", "current_ratio", "roe", "roa", "profit_margin"]
RATIO_NAMES = {"pe_ratio": "P/E ratio", "debt_to_equity": "debt-to-equity ratio", "current_ratio": "current ratio", "roe": "return on equity", "roa": "ROA", "profit_margin": "profit margin"}
CURRENCIES = ["USD", "EUR", "GBP", "JPY", "CHF", "CAD", "AUD", "INR", "CNY", "KRW"]
CIKS = {"AAPL": "0000320193", "MSFT": "0000789019", "TSLA": "0001318605", "AMZN": "0001018724", "GOOGL": "0001652044", "META": "0001326801", "JPM": "0000019617", "GS": "0000886982"}

augmented = []

# 1. get_stock_price variations
stock_templates = [
    "What's the current price of {company} stock?",
    "How much is {ticker} trading at?",
    "Get me the stock price for {company}",
    "Check {ticker} stock price on {exchange}",
    "What is {company}'s share price right now?",
    "Look up the price of {ticker}",
    "Can you check {company}'s stock price?",
    "Show me {ticker}'s current trading price",
    "I want to know {company}'s stock price",
    "What's {ticker} at right now?",
    "Pull up the stock price for {company} on {exchange}",
    "How is {ticker} performing today?",
]
for _ in range(80):
    t = random.choice(TICKERS)
    ex = random.choice(["NYSE", "NASDAQ"])
    template = random.choice(stock_templates)
    query = template.format(company=COMPANY_NAMES[t], ticker=t, exchange=ex)
    call = {"function": "get_stock_price", "arguments": {"ticker": t, "exchange": ex}}
    augmented.append({"instruction": INSTRUCTION, "input": query, "output": json.dumps(call)})

# 2. get_financial_report variations
report_templates = [
    "Pull up {company}'s {report_type} for {year}",
    "Get me {ticker}'s {report_type} filing from {year}",
    "I need {company}'s {report_type} report for fiscal year {year}",
    "Show me the {report_type} for {ticker}, year {year}",
    "Can you retrieve {company}'s {report_type} for {year}?",
    "Find {ticker}'s annual {report_type} from {year}",
    "Look up {company}'s {report_type} filing for {year}",
    "Get the {report_type} report for {company} from {year}",
]
for _ in range(60):
    t = random.choice(TICKERS)
    rt = random.choice(REPORT_TYPES)
    yr = random.choice(YEARS)
    template = random.choice(report_templates)
    query = template.format(company=COMPANY_NAMES[t], ticker=t, report_type=rt, year=yr)
    call = {"function": "get_financial_report", "arguments": {"company": t, "report_type": rt, "year": yr}}
    if rt == "10-Q":
        q = random.choice(QUARTERS)
        call["arguments"]["quarter"] = q
        query += f" Q{q}"
    augmented.append({"instruction": INSTRUCTION, "input": query, "output": json.dumps(call)})

# 3. calculate_ratio variations
ratio_templates = [
    "Calculate the {ratio_name} with {desc_num} of ${num} and {desc_den} of ${den}",
    "What's the {ratio_name} if {desc_num} is ${num} and {desc_den} is ${den}?",
    "Compute {ratio_name}: {desc_num} ${num}, {desc_den} ${den}",
    "Find the {ratio_name} given {desc_num}=${num} and {desc_den}=${den}",
    "I need the {ratio_name} — {desc_num} is ${num}B, {desc_den} is ${den}B",
]
ratio_descs = {
    "pe_ratio": ("stock price", "earnings per share"),
    "debt_to_equity": ("total debt", "total equity"),
    "current_ratio": ("current assets", "current liabilities"),
    "roe": ("net income", "shareholder equity"),
    "roa": ("net income", "total assets"),
    "profit_margin": ("net income", "revenue"),
}
for _ in range(60):
    rt = random.choice(RATIO_TYPES)
    num = round(random.uniform(5, 500), 2)
    den = round(random.uniform(5, 500), 2)
    desc_num, desc_den = ratio_descs[rt]
    template = random.choice(ratio_templates)
    query = template.format(ratio_name=RATIO_NAMES[rt], desc_num=desc_num, desc_den=desc_den, num=num, den=den)
    call = {"function": "calculate_ratio", "arguments": {"ratio_type": rt, "numerator": num, "denominator": den}}
    augmented.append({"instruction": INSTRUCTION, "input": query, "output": json.dumps(call)})

# 4. search_sec_filings variations
sec_templates = [
    "Search SEC for {query} in {filing_type} filings",
    "Find SEC filings about {query}",
    "Look up {query} disclosures in {company}'s {filing_type}",
    "Search EDGAR for {query} from {date_from} to {date_to}",
    "Find all {filing_type} filings mentioning {query}",
    "Search SEC for {company}'s {query} disclosures",
]
sec_queries = ["climate risk", "cybersecurity", "AI investments", "executive compensation", "risk factors", "supply chain", "ESG", "insider trading", "revenue recognition", "goodwill impairment"]
for _ in range(50):
    sq = random.choice(sec_queries)
    ft = random.choice(["10-K", "10-Q"])
    t = random.choice(list(CIKS.keys()))
    yr = random.choice(YEARS)
    template = random.choice(sec_templates)
    args = {"query": sq}
    if "{company}" in template:
        args["company_cik"] = CIKS[t]
    if "{filing_type}" in template:
        args["filing_type"] = ft
    if "{date_from}" in template:
        args["date_from"] = f"{yr}-01-01"
        args["date_to"] = f"{yr}-12-31"
    query = template.format(query=sq, filing_type=ft, company=COMPANY_NAMES[t], date_from=f"{yr}-01-01", date_to=f"{yr}-12-31")
    call = {"function": "search_sec_filings", "arguments": args}
    augmented.append({"instruction": INSTRUCTION, "input": query, "output": json.dumps(call)})

# 5. convert_currency variations
currency_templates = [
    "Convert {amount} {from_c} to {to_c}",
    "How much is {amount} {from_c} in {to_c}?",
    "Change {amount} {from_c} into {to_c}",
    "What's {amount} {from_c} worth in {to_c}?",
    "I need to convert {amount} {from_c} to {to_c}",
    "Exchange {amount} {from_c} for {to_c}",
]
for _ in range(50):
    fc = random.choice(CURRENCIES)
    tc = random.choice([c for c in CURRENCIES if c != fc])
    amt = random.choice([100, 500, 1000, 5000, 10000, 50000, 100000, 250000, 500000, 1000000])
    template = random.choice(currency_templates)
    query = template.format(amount=amt, from_c=fc, to_c=tc)
    call = {"function": "convert_currency", "arguments": {"amount": amt, "from_currency": fc, "to_currency": tc}}
    augmented.append({"instruction": INSTRUCTION, "input": query, "output": json.dumps(call)})

# 6. Refusal examples (no matching function)
refusal_queries = [
    "Send an email to the CFO about the quarterly results",
    "Schedule a meeting with the investment team for next Tuesday",
    "Order lunch for the trading desk",
    "Book a flight to the investor conference",
    "Print the financial report",
    "Call my broker and place a trade",
    "Set a reminder for the earnings call",
    "Draft a memo about the merger",
    "Update the CRM with the client's new contact info",
    "Post the results on social media",
    "Create a PowerPoint presentation of the financials",
    "Play me a song",
    "What's the weather like today?",
    "Write a poem about stocks",
    "Tell me a joke about finance",
    "Turn off the lights in the office",
    "Order new monitors for the trading floor",
    "Reserve a conference room for 3pm",
    "Send a Slack message to the team",
    "Upload the spreadsheet to SharePoint",
]
for q in refusal_queries:
    call = {"function": "NONE", "reason": "No available function can handle this request. Available functions are limited to stock prices, financial reports, ratio calculations, SEC filing searches, and currency conversion."}
    augmented.append({"instruction": INSTRUCTION, "input": q, "output": json.dumps(call)})

random.shuffle(augmented)

# Split 90/10
split = int(len(augmented) * 0.9)
train_data = augmented[:split]
eval_data = augmented[split:]

Path("data").mkdir(exist_ok=True)
for name, data in [("data/augmented_sft_train.jsonl", train_data), ("data/augmented_sft_eval.jsonl", eval_data)]:
    with open(name, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

print(f"Generated {len(augmented)} augmented examples")
print(f"  Train: {len(train_data)}, Eval: {len(eval_data)}")
print(f"  Saved to data/augmented_sft_train.jsonl and data/augmented_sft_eval.jsonl")
print(f"\nBreakdown:")
from collections import Counter
funcs = [json.loads(e["output"])["function"] for e in augmented]
for fn, count in Counter(funcs).most_common():
    print(f"  {fn}: {count}")
print(f"\n--- Sample ---")
for ex in train_data[:3]:
    print(f"  Q: {ex['input'][:60]}")
    print(f"  A: {ex['output'][:80]}")
    print()

Generated 320 augmented examples
  Train: 288, Eval: 32
  Saved to data/augmented_sft_train.jsonl and data/augmented_sft_eval.jsonl

Breakdown:
  get_stock_price: 80
  get_financial_report: 60
  calculate_ratio: 60
  convert_currency: 50
  search_sec_filings: 50
  NONE: 20

--- Sample ---
  Q: Get the earnings report for Meta from 2024
  A: {"function": "get_financial_report", "arguments": {"company": "META", "report_ty

  Q: Turn off the lights in the office
  A: {"function": "NONE", "reason": "No available function can handle this request. A

  Q: Convert 5000 AUD to CHF
  A: {"function": "convert_currency", "arguments": {"amount": 5000, "from_currency": 



In [8]:
# Download Glaive dataset and combine with augmented data
%cd {PROJECT_DIR}
!python src/download_dataset.py --max-examples 10000

# Count how many Glaive examples have valid JSON function calls
glaive_fc_train = []
glaive_fc_eval = []
for name, dest in [("data/glaive_sft_train.jsonl", glaive_fc_train), ("data/glaive_sft_eval.jsonl", glaive_fc_eval)]:
    with open(name) as f:
        for line in f:
            ex = json.loads(line)
            try:
                parsed = json.loads(ex["output"])
                if isinstance(parsed, dict) and "function" in parsed:
                    dest.append(ex)
            except (json.JSONDecodeError, KeyError):
                pass

print(f"Glaive function-call examples: {len(glaive_fc_train)} train, {len(glaive_fc_eval)} eval")

# Load augmented data
aug_train = [json.loads(l) for l in open("data/augmented_sft_train.jsonl")]
aug_eval = [json.loads(l) for l in open("data/augmented_sft_eval.jsonl")]
print(f"Augmented examples: {len(aug_train)} train, {len(aug_eval)} eval")

# Combine
combined_train = aug_train + glaive_fc_train
combined_eval = aug_eval + glaive_fc_eval
random.shuffle(combined_train)

for name, data in [("data/combined_sft_train.jsonl", combined_train), ("data/combined_sft_eval.jsonl", combined_eval)]:
    with open(name, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

print(f"\nCombined dataset: {len(combined_train)} train, {len(combined_eval)} eval")

/content
Processing 10000 examples...
Prepared 205 valid SFT examples
Saved 184 examples to data/glaive_sft_train.jsonl
Saved 21 examples to data/glaive_sft_eval.jsonl

To use this data, update configs/sft_config.yaml:
  train_path: "data/glaive_sft_train.jsonl"
  eval_path: "data/glaive_sft_eval.jsonl"
Glaive function-call examples: 184 train, 21 eval
Augmented examples: 288 train, 32 eval

Combined dataset: 472 train, 53 eval


---
## 4. Run Unit Tests

In [9]:
%cd {PROJECT_DIR}
!pip install -q pytest
!python -m pytest tests/ -v

/content
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/fine-tuning-lora-dpo
configfile: pyproject.toml
plugins: typeguard-4.5.2, langsmith-0.10.2, anyio-4.14.2
collected 16 items                                                             

tests/test_configs.py::test_sft_config_valid PASSED                      [  6%]
tests/test_configs.py::test_dpo_config_valid PASSED                      [ 12%]
tests/test_evaluate.py::test_evaluate_valid_json PASSED                  [ 18%]
tests/test_evaluate.py::test_evaluate_invalid_json PASSED                [ 25%]
tests/test_evaluate.py::test_evaluate_valid_json_invalid_schema PASSED   [ 31%]
tests/test_evaluate.py::test_function_call_schema PASSED                 [ 37%]
tests/test_evaluate.py::test_function_call_refusal PASSED                [ 43%]
tests/test_evaluate.py::tes

---
## 5. Test Evaluation Logic (No Training Needed)

In [10]:

import sys
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from src.evaluate import evaluate_json_output, evaluate_function_call

# Test 1: Valid JSON + correct function call
good = json.dumps({"function": "get_stock_price", "arguments": {"ticker": "GOOGL"}})
result = evaluate_function_call(good, "get_stock_price", {"ticker": "GOOGL"})
print(f"Test 1 — Correct call:     {result}")
assert result["json_valid"] and result["function_correct"] and result["args_correct"]

# Test 2: Wrong function selected
bad = json.dumps({"function": "convert_currency", "arguments": {"ticker": "GOOGL"}})
result = evaluate_function_call(bad, "get_stock_price", {"ticker": "GOOGL"})
print(f"Test 2 — Wrong function:   {result}")
assert result["json_valid"] and not result["function_correct"]

# Test 3: Invalid JSON
result = evaluate_json_output("not json at all")
print(f"Test 3 — Invalid JSON:     {result}")
assert not result["json_valid"]

# Test 4: Valid JSON but wrong schema
result = evaluate_json_output(json.dumps({"foo": "bar"}))
print(f"Test 4 — Bad schema:       {result}")
assert result["json_valid"] and not result["schema_valid"]

# Test 5: Correct refusal
refusal = json.dumps({"function": "NONE", "reason": "Cannot do that"})
result = evaluate_function_call(refusal, "NONE")
print(f"Test 5 — Refusal:          {result}")
assert result["function_correct"]

print("\nAll evaluation tests passed!")

Test 1 — Correct call:     {'json_valid': True, 'schema_valid': True, 'function_correct': True, 'args_correct': True}
Test 2 — Wrong function:   {'json_valid': True, 'schema_valid': True, 'function_correct': False, 'args_correct': False}
Test 3 — Invalid JSON:     {'json_valid': False, 'schema_valid': False, 'function_correct': False, 'args_correct': False}
Test 4 — Bad schema:       {'json_valid': True, 'schema_valid': False, 'function_correct': False, 'args_correct': False}
Test 5 — Refusal:          {'json_valid': True, 'schema_valid': True, 'function_correct': True, 'args_correct': False}

All evaluation tests passed!


---
## 6. Phase 1 — Supervised Fine-Tuning (SFT) with QLoRA

This trains LoRA adapters on top of Qwen 3 8B using 4-bit quantization.

**Expected time:** ~5-10 min on T4 with the small custom dataset (20 examples).

In [11]:
os.chdir(PROJECT_DIR)
# Optional: disable W&B if you don't have an account
os.environ["WANDB_DISABLED"] = "true"
# If you want W&B logging, comment the line above and run:
# !wandb login

In [12]:
import yaml
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

# Load config
with open("configs/sft_config.yaml") as f:
    config = yaml.safe_load(f)

# Override to use combined dataset (augmented financial + Glaive function calls)
config["data"]["train_path"] = "data/combined_sft_train.jsonl"
config["data"]["eval_path"] = "data/combined_sft_eval.jsonl"
config["training"]["num_epochs"] = 3
config["training"]["save_steps"] = 500
config["training"]["eval_steps"] = 100
config["training"]["logging_steps"] = 10

print("SFT Config:")
print(f"  Model: {config['model']['name']}")
print(f"  QLoRA (4-bit): {config['model'].get('load_in_4bit', False)}")
print(f"  LoRA rank: {config['lora']['r']}")
print(f"  Epochs: {config['training']['num_epochs']}")
print(f"  Batch size: {config['training']['per_device_train_batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Train data: {config['data']['train_path']}")
print(f"  Eval data: {config['data']['eval_path']}")

SFT Config:
  Model: Qwen/Qwen3-8B
  QLoRA (4-bit): True
  LoRA rank: 16
  Epochs: 3
  Batch size: 4
  Learning rate: 0.0002
  Train data: data/combined_sft_train.jsonl
  Eval data: data/combined_sft_eval.jsonl


In [13]:
# Free any leftover GPU memory from previous runs
import gc
try:
    del model
except NameError:
    pass
try:
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory before loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Setup quantization
bnb_config = None
if config["model"].get("load_in_4bit"):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model_name = config["model"]["name"]
print(f"Loading tokenizer from {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model from {model_name} (this may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

if bnb_config:
    model = prepare_model_for_kbit_training(model)

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU memory before loading: 0.00 GB
Loading tokenizer from Qwen/Qwen3-8B...


Loading model from Qwen/Qwen3-8B (this may take a few minutes)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded. Parameters: 8,190,735,360
GPU memory used: 8.58 GB


In [14]:
# Configure LoRA
lora_cfg = config["lora"]
peft_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)
print(f"LoRA config: rank={lora_cfg['r']}, alpha={lora_cfg['lora_alpha']}, target_modules={lora_cfg['target_modules']}")

# Load dataset
dataset = load_dataset("json", data_files={
    "train": config["data"]["train_path"],
    "eval": config["data"]["eval_path"],
})
print(f"Train examples: {len(dataset['train'])}, Eval examples: {len(dataset['eval'])}")

def format_example(example):
    return f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Output:\n{example['output']}"

# Show a formatted example
print("\n--- Formatted training example ---")
print(format_example(dataset["train"][0]))

LoRA config: rank=16, alpha=32, target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Train examples: 472, Eval examples: 53

--- Formatted training example ---
### Instruction:
You have access to the following functions. Call the appropriate function based on the user's request.

### Input:
What's 500 GBP worth in INR?

### Output:
{"function": "convert_currency", "arguments": {"amount": 500, "from_currency": "GBP", "to_currency": "INR"}}


In [15]:
# Train!
train_cfg = config["training"]
training_args = TrainingArguments(
    output_dir=config["output"]["dir"],
    num_train_epochs=train_cfg["num_epochs"],
    per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
    gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
    learning_rate=train_cfg["learning_rate"],
    warmup_ratio=train_cfg["warmup_ratio"],
    logging_steps=train_cfg["logging_steps"],
    save_steps=train_cfg["save_steps"],
    eval_steps=train_cfg["eval_steps"],
    eval_strategy="steps",
    bf16=True,
    report_to="none",  # Change to "wandb" if you want W&B logging
)

# TRL >= 0.15 renamed max_seq_length to max_length
import inspect
sft_kwargs = {}
sft_sig = inspect.signature(SFTTrainer.__init__)
if "max_seq_length" in sft_sig.parameters:
    sft_kwargs["max_seq_length"] = train_cfg["max_seq_length"]
elif "max_length" in sft_sig.parameters:
    sft_kwargs["max_length"] = train_cfg["max_seq_length"]

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    peft_config=peft_config,
    formatting_func=format_example,
    **sft_kwargs,
)

print("Starting SFT training...")
trainer.train()
print("SFT training complete!")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/472 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/472 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/472 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/472 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting SFT training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
90,0.156110,0.181640,0.160486,125181.000000,0.942406


SFT training complete!


In [16]:
# Save the SFT model
sft_output_dir = config["output"]["dir"]
trainer.save_model(sft_output_dir)
tokenizer.save_pretrained(sft_output_dir)
print(f"SFT model saved to {sft_output_dir}")
%cd {PROJECT_DIR}
!ls -la {sft_output_dir}

SFT model saved to models/sft-lora
/content
total 96504
-rw------- 1 root root     1092 Jul 20 21:43 adapter_config.json
-rw------- 1 root root 87361592 Jul 20 21:43 adapter_model.safetensors
-rw------- 1 root root     4168 Jul 20 21:43 chat_template.jinja
drwx------ 2 root root     4096 Jul 20 21:01 checkpoint-2
drwx------ 2 root root     4096 Jul 20 19:33 checkpoint-282
drwx------ 2 root root     4096 Jul 15 07:03 checkpoint-3
drwx------ 2 root root     4096 Jul 20 21:26 checkpoint-54
drwx------ 2 root root     4096 Jul 20 21:43 checkpoint-90
-rw------- 1 root root     1426 Jul 20 21:43 README.md
-rw------- 1 root root      691 Jul 20 21:43 tokenizer_config.json
-rw------- 1 root root 11422650 Jul 20 21:43 tokenizer.json
-rw------- 1 root root     5713 Jul 20 21:43 training_args.bin


---
## 7. Test SFT Model — Inference Before DPO

Let's see how the SFT model performs before we apply DPO.

In [17]:
from peft import PeftModel
import re

def run_inference(model, tokenizer, user_input, max_new_tokens=256):
    """Run inference with the instruction format used during training."""
    prompt = (
        f"### Instruction:\nYou have access to the following functions. "
        f"Call the appropriate function based on the user's request.\n\n"
        f"### Input:\n{user_input}\n\n"
        f"### Output:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return extract_first_json(response.strip())

def extract_first_json(text):
    """Extract the first valid JSON object from model output (ignores trailing rambling)."""
    # Find the first { and match its closing }
    start = text.find("{")
    if start == -1:
        return text.strip()
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return text[start:]  # unclosed, return what we have

# Test cases
test_inputs = [
    "What's the current price of Google stock?",
    "Pull up Amazon's 10-K for 2024",
    "Calculate ROA with net income $15B and total assets $350B",
    "Convert 500000 EUR to JPY",
    "Order lunch for the trading desk",  # Should refuse
    "Search SEC for climate disclosures in 10-K filings since 2024",
]

print("=" * 60)
print("SFT Model Inference Results")
print("=" * 60)
for inp in test_inputs:
    output = run_inference(model, tokenizer, inp)
    print(f"\nInput:  {inp}")
    print(f"Output: {output}")
    try:
        parsed = json.loads(output)
        print(f"Valid JSON: Yes | Function: {parsed.get('function', 'N/A')}")
    except json.JSONDecodeError:
        print(f"Valid JSON: No")
    print("-" * 60)

SFT Model Inference Results


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Input:  What's the current price of Google stock?
Output: {"function": "get_stock_price", "arguments": {"ticker": "GOOGL", "exchange": "NYSE"}}
Valid JSON: Yes | Function: get_stock_price
------------------------------------------------------------

Input:  Pull up Amazon's 10-K for 2024
Output: {"function": "get_financial_report", "arguments": {"company": "AMZN", "report_type": "10-K", "year": 2024}}
Valid JSON: Yes | Function: get_financial_report
------------------------------------------------------------

Input:  Calculate ROA with net income $15B and total assets $350B
Output: {"function": "calculate_ratio", "arguments": {"ratio_type": "roa", "numerator": 15, "denominator": 350}}
Valid JSON: Yes | Function: calculate_ratio
------------------------------------------------------------

Input:  Convert 500000 EUR to JPY
Output: {"function": "convert_currency", "arguments": {"amount": 500000, "from_currency": "EUR", "to_currency": "JPY"}}
Valid JSON: Yes | Function: convert_currency

---
## 8. Phase 2 — DPO Preference Optimization

Train the model to prefer correct function calls over incorrect ones.

In [18]:
# Free up memory from SFT training
try:
    del model, trainer
except NameError:
    pass
try:
    del dpo_model, dpo_trainer
except NameError:
    pass
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU memory after cleanup: 0.02 GB


In [19]:
from trl import DPOTrainer

# Load DPO config
with open("configs/dpo_config.yaml") as f:
    dpo_config = yaml.safe_load(f)

print("DPO Config:")
print(f"  SFT checkpoint: {dpo_config['model']['sft_checkpoint']}")
print(f"  Beta: {dpo_config['dpo']['beta']}")
print(f"  Loss type: {dpo_config['dpo']['loss_type']}")
print(f"  Epochs: {dpo_config['training']['num_epochs']}")
print(f"  Learning rate: {dpo_config['training']['learning_rate']}")

DPO Config:
  SFT checkpoint: models/sft-lora
  Beta: 0.1
  Loss type: sigmoid
  Epochs: 1
  Learning rate: 5e-05


In [20]:
sft_path = dpo_config["model"]["sft_checkpoint"]

# Setup quantization for DPO
bnb_config = None
if dpo_config["model"].get("load_in_4bit"):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

print(f"Loading SFT checkpoint from {sft_path}...")
dpo_tokenizer = AutoTokenizer.from_pretrained(sft_path)
dpo_model = AutoModelForCausalLM.from_pretrained(
    sft_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading SFT checkpoint from models/sft-lora...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Model loaded. GPU memory: 6.52 GB


In [21]:
# Load DPO dataset
dpo_dataset = load_dataset("json", data_files={
    "train": dpo_config["data"]["train_path"],
    "eval": dpo_config["data"]["eval_path"],
})
print(f"DPO Train examples: {len(dpo_dataset['train'])}, Eval: {len(dpo_dataset['eval'])}")

# Show a preference pair
print("\n--- Sample DPO pair ---")
sample = dpo_dataset["train"][0]
print(f"Prompt:   {sample['prompt']}")
print(f"Chosen:   {sample['chosen']}")
print(f"Rejected: {sample['rejected']}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

DPO Train examples: 6, Eval: 2

--- Sample DPO pair ---
Prompt:   You have access to financial tools. User says: 'What is Apple's stock price?'
Chosen:   {"function": "get_stock_price", "arguments": {"ticker": "AAPL", "exchange": "NASDAQ"}}
Rejected: {"function": "get_stock_price", "arguments": {"ticker": "Apple"}}


In [22]:
# Train DPO
# DPO normally loads TWO copies of the model (policy + reference), which OOMs on T4.
# We use precompute_ref_log_probs=True to compute reference logprobs first, then free
# the ref model before training — fits in T4's 15GB VRAM.

dpo_train_cfg = dpo_config["training"]
dpo_cfg = dpo_config["dpo"]

import inspect
dpo_sig = inspect.signature(DPOTrainer.__init__)
dpo_params = set(dpo_sig.parameters.keys())

training_args_kwargs = dict(
    output_dir=dpo_config["output"]["dir"],
    num_train_epochs=dpo_train_cfg["num_epochs"],
    per_device_train_batch_size=dpo_train_cfg["per_device_train_batch_size"],
    gradient_accumulation_steps=dpo_train_cfg["gradient_accumulation_steps"],
    learning_rate=dpo_train_cfg["learning_rate"],
    warmup_ratio=dpo_train_cfg["warmup_ratio"],
    logging_steps=dpo_train_cfg["logging_steps"],
    save_steps=dpo_train_cfg["save_steps"],
    bf16=True,
    report_to="none",
)

try:
    from trl import DPOConfig
    dpo_training_args = DPOConfig(
        **training_args_kwargs,
        beta=dpo_cfg["beta"],
        loss_type=dpo_cfg["loss_type"],
        max_length=512,  # Reduced from 2048 to save VRAM
        precompute_ref_log_probs=True,  # Precompute ref logprobs to avoid loading 2 models at once
    )
    trainer_kwargs = {}
except ImportError:
    dpo_training_args = TrainingArguments(**training_args_kwargs)
    trainer_kwargs = dict(
        beta=dpo_cfg["beta"],
        loss_type=dpo_cfg["loss_type"],
        max_length=512,
        precompute_ref_log_probs=True,
    )

tok_key = "processing_class" if "processing_class" in dpo_params else "tokenizer"

dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_training_args,
    train_dataset=dpo_dataset["train"],
    eval_dataset=dpo_dataset["eval"],
    **{tok_key: dpo_tokenizer},
    **trainer_kwargs,
)

print("Starting DPO training...")
dpo_trainer.train()
print("DPO training complete!")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting DPO training...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DPO training complete!


In [23]:
# Save DPO model
dpo_output_dir = dpo_config["output"]["dir"]
dpo_trainer.save_model(dpo_output_dir)
dpo_tokenizer.save_pretrained(dpo_output_dir)
print(f"DPO model saved to {dpo_output_dir}")
%cd {PROJECT_DIR}
!ls -la {dpo_output_dir}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DPO model saved to models/dpo-lora
/content
total 96490
-rw------- 1 root root     1092 Jul 20 21:45 adapter_config.json
-rw------- 1 root root 87361592 Jul 20 21:45 adapter_model.safetensors
-rw------- 1 root root     4168 Jul 20 21:45 chat_template.jinja
drwx------ 2 root root     4096 Jul 20 21:45 checkpoint-1
-rw------- 1 root root      188 Jul 20 21:45 generation_config.json
-rw------- 1 root root     2284 Jul 20 21:45 README.md
-rw------- 1 root root      690 Jul 20 21:45 tokenizer_config.json
-rw------- 1 root root 11422650 Jul 20 21:45 tokenizer.json
-rw------- 1 root root     5841 Jul 20 21:45 training_args.bin


---
## 9. Evaluation — Base vs SFT vs DPO

In [24]:
from src.evaluate import evaluate_function_call, evaluate_json_output, TEST_CASES

def evaluate_model(model, tokenizer, label):
    """Run all test cases and compute metrics."""
    results = []
    for tc in TEST_CASES:
        output = run_inference(model, tokenizer, tc["input"])
        metrics = evaluate_function_call(
            output,
            tc["expected_function"],
            tc.get("expected_args"),
        )
        results.append({"input": tc["input"], "output": output, **metrics})

    n = len(results)
    json_valid = sum(1 for r in results if r["json_valid"]) / n
    schema_valid = sum(1 for r in results if r["schema_valid"]) / n
    func_correct = sum(1 for r in results if r["function_correct"]) / n
    args_correct = sum(1 for r in results if r["args_correct"]) / n

    print(f"\n{'=' * 50}")
    print(f"{label} Results ({n} test cases)")
    print(f"{'=' * 50}")
    print(f"  JSON Validity:     {json_valid:.1%}")
    print(f"  Schema Validity:   {schema_valid:.1%}")
    print(f"  Function Accuracy: {func_correct:.1%}")
    print(f"  Argument Accuracy: {args_correct:.1%}")

    print(f"\nDetailed results:")
    for r in results:
        status = "PASS" if r["function_correct"] else "FAIL"
        print(f"  [{status}] {r['input'][:50]}...")
        print(f"        -> {r['output'][:80]}")

    return {"json_valid": json_valid, "schema_valid": schema_valid,
            "func_correct": func_correct, "args_correct": args_correct}

# Free DPO model and load SFT model for evaluation
# (DPO with only 6 examples degraded the model — need more data for DPO to help)
del dpo_model, dpo_trainer
gc.collect()
torch.cuda.empty_cache()

print("Loading SFT model for evaluation...")
sft_eval_tokenizer = AutoTokenizer.from_pretrained("models/sft-lora")
sft_eval_model = AutoModelForCausalLM.from_pretrained(
    "models/sft-lora",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

sft_metrics = evaluate_model(sft_eval_model, sft_eval_tokenizer, "SFT Fine-Tuned (QLoRA)")

print("\n" + "=" * 50)
print("Note on DPO")
print("=" * 50)
print("DPO training with only 6 preference pairs degraded the model.")
print("This is expected — DPO needs hundreds/thousands of pairs to be")
print("effective. With more data (e.g., Glaive dataset + generated")
print("preference pairs), DPO would improve over SFT.")

Loading SFT model for evaluation...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



SFT Fine-Tuned (QLoRA) Results (6 test cases)
  JSON Validity:     100.0%
  Schema Validity:   100.0%
  Function Accuracy: 100.0%
  Argument Accuracy: 83.3%

Detailed results:
  [PASS] What's the current price of Google stock?...
        -> {"function": "get_stock_price", "arguments": {"ticker": "GOOGL", "exchange": "NY
  [PASS] Pull up Amazon's 10-K for 2024...
        -> {"function": "get_financial_report", "arguments": {"company": "AMZN", "report_ty
  [PASS] Calculate ROA with net income $15B and total asset...
        -> {"function": "calculate_ratio", "arguments": {"ratio_type": "roa", "numerator": 
  [PASS] Convert 500000 EUR to JPY...
        -> {"function": "convert_currency", "arguments": {"amount": 500000, "from_currency"
  [PASS] Order lunch for the trading desk...
        -> {"function": "NONE", "reason": "No available function can handle this request. A
  [PASS] Search SEC for climate disclosures in 10-K filings...
        -> {"function": "search_sec_filings", "arguments"

---
## 10. Interactive Testing

Try your own queries against the fine-tuned model.

In [25]:
# Change this to test different inputs!
test_queries = [
    "What is the stock price of NVIDIA?",
    "Get me Meta's 10-Q for Q2 2025",
    "Calculate the P/E ratio: stock is $450, EPS is $22.50",
    "Convert 75000 GBP to INR",
    "Play me a song",  # Should refuse
    "Search SEC for insider trading reports from 2025",
]

print("=" * 60)
print("Interactive Testing — SFT Fine-Tuned Model")
print("=" * 60)
for query in test_queries:
    output = run_inference(sft_eval_model, sft_eval_tokenizer, query)
    print(f"\nQuery:  {query}")
    print(f"Output: {output}")
    try:
        parsed = json.loads(output)
        print(f"  -> Function: {parsed.get('function', 'N/A')}")
        if 'arguments' in parsed:
            print(f"  -> Args: {json.dumps(parsed['arguments'])}")
        if 'reason' in parsed:
            print(f"  -> Reason: {parsed['reason']}")
    except json.JSONDecodeError:
        print("  -> (not valid JSON)")
    print("-" * 60)

Interactive Testing — SFT Fine-Tuned Model

Query:  What is the stock price of NVIDIA?
Output: {"function": "get_stock_price", "arguments": {"ticker": "NVDA", "exchange": "NASDAQ"}}
  -> Function: get_stock_price
  -> Args: {"ticker": "NVDA", "exchange": "NASDAQ"}
------------------------------------------------------------

Query:  Get me Meta's 10-Q for Q2 2025
Output: {"function": "get_financial_report", "arguments": {"company": "META", "report_type": "10-Q", "year": 2025, "quarter": 2}}
  -> Function: get_financial_report
  -> Args: {"company": "META", "report_type": "10-Q", "year": 2025, "quarter": 2}
------------------------------------------------------------

Query:  Calculate the P/E ratio: stock is $450, EPS is $22.50
Output: {"function": "calculate_ratio", "arguments": {"ratio_type": "pe_ratio", "numerator": 450, "denominator": 22.50}}
  -> Function: calculate_ratio
  -> Args: {"ratio_type": "pe_ratio", "numerator": 450, "denominator": 22.5}
---------------------------------

---
## 11. Summary & Saved Artifacts

Everything is saved back to your Google Drive:

In [26]:
print("Saved artifacts on Google Drive:")
print(f"  Project dir:     {PROJECT_DIR}")
print(f"  Training data:   {PROJECT_DIR}/data/")
print(f"  SFT checkpoint:  {PROJECT_DIR}/models/sft-lora/")
print(f"  DPO checkpoint:  {PROJECT_DIR}/models/dpo-lora/")
print()
print("To reuse the fine-tuned model later:")
print("  from peft import PeftModel")
print("  model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-8B', ...)")
print("  model = PeftModel.from_pretrained(model, 'models/dpo-lora')")
print()

# Show model sizes
import subprocess
for d in ["models/sft-lora", "models/dpo-lora"]:
    if os.path.exists(d):
        size = subprocess.run(["du", "-sh", d], capture_output=True, text=True).stdout.strip()
        print(f"  {size}")

Saved artifacts on Google Drive:
  Project dir:     /content/drive/MyDrive/fine-tuning-lora-dpo
  Training data:   /content/drive/MyDrive/fine-tuning-lora-dpo/data/
  SFT checkpoint:  /content/drive/MyDrive/fine-tuning-lora-dpo/models/sft-lora/
  DPO checkpoint:  /content/drive/MyDrive/fine-tuning-lora-dpo/models/dpo-lora/

To reuse the fine-tuned model later:
  from peft import PeftModel
  model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-8B', ...)
  model = PeftModel.from_pretrained(model, 'models/dpo-lora')

  1.4G	models/sft-lora
  189M	models/dpo-lora
